# Hybrid Search
Hybrid search in RAG combines **vector (semantic) search** with **keyword (lexical) search** to retrieve documents.
Vector search captures meaning and context, while keyword search ensures exact terms, names, and numbers aren’t missed.
This is important because embeddings can miss rare words, acronyms, or exact matches that users expect.
Hybrid search improves recall and precision, making RAG answers more accurate and reliable in real-world queries.


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("PINECONE_API_KEY")

In [17]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-rag"
pc = Pinecone(api_key = api_key)


In [18]:
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384 ,
        metric="dotproduct",
        spec = ServerlessSpec(cloud="aws",region="us-east-1"),
    )

In [19]:
index = pc.Index(index_name)
index

d:\langchain\lenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [21]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [26]:
sentences = [
    "Machine learning helps computers learn patterns from data.",
    "LangChain is used to build applications with large language models.",
    "Pinecone is a vector database for fast similarity search.",
    "Embeddings convert text into numerical representations.",
    "FastAPI is commonly used to build backend APIs in Python."
]

bm25_encoder.fit(sentences)
bm25_encoder.dump("bm25_values.json")
bm25_encoder=BM25Encoder().load("bm25_values.json")

100%|██████████| 5/5 [00:00<00:00, 22.09it/s]


In [27]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)

In [28]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x0000021366122570>, index=<pinecone.db_data.index.Index object at 0x000002134B6BA930>)

In [29]:
retriever.add_texts([
    "Machine learning helps computers learn patterns from data.",
    "LangChain is used to build applications with large language models.",
    "Pinecone is a vector database for fast similarity search.",
    "Embeddings convert text into numerical representations.",
    "FastAPI is commonly used to build backend APIs in Python."
])

100%|██████████| 1/1 [00:06<00:00,  6.37s/it]


In [31]:
retriever.invoke("what is fastapi?")

[Document(metadata={'score': 0.519364834}, page_content='FastAPI is commonly used to build backend APIs in Python.'),
 Document(metadata={'score': 0.104645729}, page_content='LangChain is used to build applications with large language models.'),
 Document(metadata={'score': 0.0674643517}, page_content='Pinecone is a vector database for fast similarity search.'),
 Document(metadata={'score': 0.0229697227}, page_content='Embeddings convert text into numerical representations.')]